# Module 12 — All five failure modes, on demand

**THE ONE IDEA:** you cannot ask a good model to misbehave. Ask it to loop forever and it
won't. So the failures get **scripted** — and that is not a testing shortcut, it **is**
the technique.

> **A guard you cannot trigger on demand is a guard you have never verified.**

`_fake_model.py` has the same interface as the OpenAI client, but its turns come from a
list you wrote. Every module from here on depends on it.

| | failure | what you see |
|---|---|---|
| FM1 | tools-as-text | JSON in `content`, `tool_calls` is `None` |
| FM2 | composition collapse | a prose plan, zero tool calls |
| FM3 | post-success wander | correct at step N, still going at N+3 |
| FM4 | infinite loop | same call, same args, forever |
| FM5 | hallucinated tool/args | `lookup_acct`, or a made-up argument name |


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _fake_model import FakeModel, tool_turn, text_turn, raw_turn
from _tools import run_tool, REGISTRY

def loop(fake, max_steps=8, verbose=True):
    """The NAIVE loop from module 08. No guards at all — that is the point.
    It returns a trace so module 13 can prove its guards catch each case."""
    trace, messages = [], [{"role": "user", "content": "What is the year-2 ERC on 250000?"}]
    for step in range(1, max_steps + 1):
        r = fake.create(messages=messages)
        msg = r.choices[0].message
        fin = r.choices[0].finish_reason
        if fin != "tool_calls" or not msg.tool_calls:
            trace.append(("text", msg.content))
            if verbose: print(f"  step {step}: TEXT {str(msg.content)[:78]!r}")
            return trace
        for tc in msg.tool_calls:
            name = tc.function.name
            try:    args = json.loads(tc.function.arguments)
            except json.JSONDecodeError: args = {"__malformed__": tc.function.arguments}
            out = run_tool(name, args)
            trace.append(("tool", name, args, out))
            if verbose: print(f"  step {step}: TOOL {name}({args}) -> {out[:58]}")
    trace.append(("capped", max_steps))
    if verbose: print(f"  !! ran out of steps at {max_steps}")
    return trace

## FM1 — Tools-as-text

The model writes the tool call as JSON *inside* `content`, and leaves `tool_calls` empty.
Your orchestrator sees a normal text turn. The tool is "called" and never runs.

In [ ]:
fm1 = FakeModel([raw_turn(
    'I will look that up. {"name": "search_policy", "arguments": {"query": "erc"}}',
    None, "stop")])
print("FM1 — tools-as-text"); t1 = loop(fm1)
print("  -> loop ended thinking it had an ANSWER. No tool ran. No error raised.")

## FM2 — Multi-tool composition collapse

Two tools were needed. The model describes both beautifully and calls neither.

In [ ]:
fm2 = FakeModel([text_turn(
    "Here is my plan: first I will search the ERC policy, then I will calculate "
    "4% of 250000. That will give the year-2 charge.")])
print("FM2 — composition collapse"); t2 = loop(fm2)
print("  -> a plan is not an action. Zero tool calls executed.")

## FM3 — Post-success wander

It had the answer at step 2. The protocol offers another action slot, so it takes one.

In [ ]:
fm3 = FakeModel([
    tool_turn("search_policy", {"query": "erc"},        "c1"),
    tool_turn("calculate",     {"expression": "250000*0.04"}, "c2"),   # <- done here
    tool_turn("search_policy", {"query": "ltv"},        "c3"),          # irrelevant
    tool_turn("search_policy", {"query": "rates"},      "c4"),          # irrelevant
    text_turn("The year-2 ERC is 10000.")])
print("FM3 — post-success wander"); t3 = loop(fm3)
print("  -> 4 tool calls where 2 sufficed. Right answer, 2x the bill.")

## FM4 — Infinite loop

`FakeModel` repeats its last turn forever once the script runs out, which is exactly how
a model that cannot recognise 'I am done' behaves.

In [ ]:
fm4 = FakeModel([tool_turn("search_policy", {"query": "erc"}, "c1")])  # repeats forever
print("FM4 — infinite loop"); t4 = loop(fm4)
print("  -> identical call, identical args, until the step ceiling. Unbounded cost.")

## FM5 — Hallucinated tool name and arguments

In [ ]:
fm5 = FakeModel([
    tool_turn("lookup_acct", {"query": "erc"}, "c1"),                       # no such tool
    tool_turn("calculate", {"principal_annual_rate_pct": "250000*0.04"}, "c2"),  # wrong arg
    text_turn("Done.")])
print("FM5 — hallucinated tool / args"); t5 = loop(fm5)
print(f"  -> registry has {sorted(REGISTRY)}")
print("  -> run_tool returned structured ERRORs instead of crashing, which is right:")
print("     an unknown tool is something the MODEL can recover from next turn.")

## Summary

In [ ]:
for tag, tr in [("FM1", t1), ("FM2", t2), ("FM3", t3), ("FM4", t4), ("FM5", t5)]:
    tools = [e for e in tr if e[0] == "tool"]
    errs  = [e for e in tools if str(e[3]).startswith("ERROR")]
    print(f"{tag}: {len(tools)} tool call(s), {len(errs)} error(s), "
          f"ended={'capped' if tr[-1][0] == 'capped' else tr[-1][0]}")

print()
print("LESSON — all five ran on demand, deterministically, with no API key and no")
print("cost. None of them RAISED. Four of the five returned a plausible-looking")
print("result. That is what makes them production bugs rather than crashes.")
print()
print("Look at the naive loop again: its only protection is `max_steps`, and that")
print("one ceiling is doing all the work in FM3 and FM4.")
print()
print("Module 13 adds the guards and re-runs these exact five scripts. Every one")
print("must be caught. That is the module's pass condition, not a nice-to-have.")

---

**Next:** `13_guardrails_and_budgets.ipynb` — catch all five.